# LELA60331 Week 8 Seminar: Turning text into features

Last week we finished building logistic regression on simulated data, where each text had just two features. Now we move to **real text**: 10,000 Yelp reviews. Before we can train a classifier on them, we have to turn each review into numbers. In this sheet we:

| Part | What you do |
|---|---|
| 1 | Load the Yelp reviews and get to know them |
| 2 | Build a vocabulary of the 5000 most frequent words |
| 3 | Split the reviews into a training set and a test set |
| 4 | One-hot encode the reviews: one feature for every word in the vocabulary |
| 5 | Write matrix-vector multiplication for these one-hot features |

As before, each activity has a cell to complete (replace every `???`) followed by a ✅ **check** cell.

In [ ]:
# ▶ Run this cell first. It defines the check() helper used after each activity.
# You don't need to understand how it works: it prints ✅ when your answer is right and ❌ when it isn't yet.
def check(label, condition):
    try:
        passed = bool(condition() if callable(condition) else condition)
    except Exception as err:
        passed = False
        label += f"  (error: {type(err).__name__}: {err})"
    print(("✅ " if passed else "❌ ") + label)
    return passed

This cell sets up `dot` and `mat_vec` from Week 7. Run it.

In [ ]:
import random

def dot(a, b):
    # the dot product of two lists of the same length
    if len(a) != len(b):
        raise ValueError("the two lists must have the same length")
    total = 0
    for i in range(len(a)):
        total += a[i] * b[i]
    return total

def mat_vec(matrix, vector):
    # matrix-vector multiplication: the dot product of each row of the matrix with the vector (Week 7, Part 4)
    result = []
    for row in matrix:
        result.append(dot(row, vector))
    return result

---
# Part 1 · The Yelp reviews

Our data set is 10,000 reviews from Yelp, each labelled **negative** (1 or 2 stars) or **positive** (3 or 4 stars).

First we download the data set. A line beginning with `!` is a command for the computer, not Python: `wget` downloads a file from the web.

In [ ]:
!wget https://raw.githubusercontent.com/cbannard/lela60331_24-25/refs/heads/main/data/yelp_reviews.txt

The file is **tab-separated**: each line holds a review, then a tab, then the label. The first line is a header. `head -n2` shows the first two lines:

In [ ]:
!head -n2 yelp_reviews.txt

The text has already been lower-cased and its punctuation separated from the words by spaces, so that it is easy to split into words. (The odd `n n` is what is left of the line breaks.)

This cell reads the file into two lists: `reviews` holds the texts and `labels` the labels, so review `i` has the label `labels[i]`. Just run it.

In [ ]:
reviews = []
labels = []

with open("yelp_reviews.txt") as fp:
    lines = fp.readlines()
for line in lines[1:]:                         # skip the header line
    fields = line.rstrip().split("\t")         # [review, label]
    reviews.append(fields[0])
    labels.append(fields[1])

### Activity 1 — Getting to know the reviews

1. `n_reviews`: the number of reviews.
2. `n_positive_reviews`: the number of positive reviews. *Hint:* the labels are strings this time, so use the list method `count`.
3. `first_review_length`: the number of words in review 0. *Hint:* split it at the spaces.

Then read review 0. Does its label make sense?

In [ ]:
n_reviews = ???
n_positive_reviews = ???
first_review_length = ???
print(n_reviews, n_positive_reviews, first_review_length)      # you should get 10000 5000 118
print(labels[0], ":", reviews[0])

In [ ]:
# ✅ Check
check("n_reviews", lambda: n_reviews == 10000)
check("n_positive_reviews", lambda: n_positive_reviews == 5000)
check("first_review_length", lambda: first_review_length == 118)

Are the classes balanced? What accuracy would a model get if it always said "positive"?

---
# Part 2 · Building a vocabulary

Our features will be the words. Each review gets one feature for every word in a **vocabulary**: 1 if the word occurs in the review and 0 if it doesn't. We use the 5000 most frequent words as our vocabulary (not an ideal solution, but a convenient one).

First we **tokenise** each review, splitting it into a list of words. This is a very naive tokenisation that just splits the text at spaces. Run the cell.

In [ ]:
tokenized_sents = []
for review in reviews:
    tokenized_sents.append(review.split())

print(len(tokenized_sents))         # one list of words per review
print(tokenized_sents[0][:12])      # the first 12 words of review 0

### Activity 2 — Counting the words

Complete the loop so that `counts` becomes a **counting dictionary** (Week 3 seminar): each word is a key and its value is the number of times the word occurs across all the reviews. *Hint:* `counts.get(word, 0)` gives the count so far, or 0 if the word hasn't been seen yet.

In [ ]:
counts = {}
for sent in tokenized_sents:
    for word in sent:
        counts[word] = ???

print(len(counts))                              # the number of different words: you should get 28495
print(counts["good"], counts["bad"])            # you should get 5012 1285

In [ ]:
# ✅ Check
check("number of different words", lambda: len(counts) == 28495)
check("count of 'good'", lambda: counts["good"] == 5012)
check("count of 'bad'", lambda: counts["bad"] == 1285)

### Activity 3 — The 5000 most frequent words

To find the most frequent words, we make a list of `(count, word)` pairs and sort it from largest to smallest, as in the Week 3 seminar. Python sorts pairs by their first element, the count.

Complete `type_list`: a list of just the **words** from the first 5000 pairs. *Hint:* a list comprehension over `pairs[:5000]` that unpacks each pair into `count, word`.

In [ ]:
pairs = [(count, word) for word, count in counts.items()]
pairs = sorted(pairs, reverse=True)
print(pairs[:5])

type_list = ???
print(len(type_list))          # you should get 5000
print(type_list[:10])          # the 10 most frequent words

In [ ]:
# ✅ Check
check("5000 words", lambda: len(type_list) == 5000)
check("the most frequent word first", lambda: type_list[0] == pairs[0][1])
check("only words, not pairs", lambda: all(type(word) == str for word in type_list))

What kinds of words are most frequent? Do you think they will be useful for telling positive from negative reviews?

Each word in the vocabulary gets its own feature, and so its own **column** in our data matrix: `type_list[j]` is the word for column `j`. We also need to go the other way, from a word to its column, so we make a dictionary `word_to_index`. Run the cell.

In [ ]:
word_to_index = {}
for j in range(len(type_list)):
    word_to_index[type_list[j]] = j

print(word_to_index["good"], type_list[word_to_index["good"]])

---
# Part 3 · Training and test sets

We want to know how well our classifier does on reviews it has **never seen**. So we split the data: we train on 80% of the reviews (8000) and test on the other 20% (2000). In a real project we would also keep a third set, a **development** set, for trying out different settings; here we just use training and test.

`random.sample(sequence, k)` picks `k` different elements at random from `sequence`. We use it to pick the indices of 8000 training reviews. The test reviews are all the indices that were not picked. Run the cell.

In [ ]:
random.seed(1)        # so that everyone gets the same split
train_ints = random.sample(range(len(reviews)), int(len(reviews) * 0.8))

# The test items are all the indices that were not picked for training.
# Checking whether something is in a set is very fast.
train_set = set(train_ints)
test_ints = []
for i in range(len(reviews)):
    if i not in train_set:
        test_ints.append(i)

print(len(train_ints), len(test_ints))
print(train_ints[:5])

### Activity 4 — Labels as numbers

Our model works with labels 1 (positive) and 0 (negative), as in Week 5. Use list comprehensions to make:

1. `y_train`: the label of each training review, as 1 or 0, in the order of `train_ints`
2. `y_test`: the same for the test reviews

*Hint:* `int(labels[i] == "positive")` is 1 if review `i` is positive and 0 if not, because `int(True)` is 1 and `int(False)` is 0.

In [ ]:
y_train = ???
y_test = ???
print(len(y_train), sum(y_train))       # you should get 8000 4026
print(len(y_test), sum(y_test))         # you should get 2000 974

In [ ]:
# ✅ Check
check("y_train", lambda: len(y_train) == 8000 and sum(y_train) == 4026)
check("y_test", lambda: len(y_test) == 2000 and sum(y_test) == 974)
check("no review is in both sets", lambda: len(set(train_ints) & set(test_ints)) == 0)

---
# Part 4 · One-hot encoding

Now we turn each review into numbers. We want a matrix **M_train** with one row per training review (8000) and one column per vocabulary word (5000), and a matrix **M_test** in the same way for the 2000 test reviews. Element $x_{i,j}$ is 1 if word $j$ occurs in review $i$, and 0 otherwise. This is called **one-hot encoding**.

### Sparse rows

As a list of lists, M_train would hold 8000 × 5000 = 40 million numbers. That is very slow to work with in plain Python, and almost all of them are 0: a review contains only a small fraction of the 5000 vocabulary words. So instead we use a **sparse** representation: for each review we store only the **columns that hold a 1**, as a list of column numbers.

### Activity 5 — A tiny example

Here is a vocabulary of just 4 words and a review. Complete the loop so that it builds both:

* `full_row`: the full row of 0s and 1s
* `sparse_row`: the list of the columns that hold a 1

In [ ]:
tiny_vocab = ["good", "bad", "food", "service"]      # columns 0, 1, 2 and 3
review = "good food good".split()

full_row = [0, 0, 0, 0]
sparse_row = []
for j in range(len(tiny_vocab)):
    if ???:
        full_row[j] = 1
        sparse_row.append(???)
print(full_row)      # you should get [1, 0, 1, 0]
print(sparse_row)    # you should get [0, 2]

In [ ]:
# ✅ Check
check("full_row", lambda: full_row == [1, 0, 1, 0])
check("sparse_row", lambda: sparse_row == [0, 2])

"good" occurs twice in the review, but its column holds a 1, not a 2: we only record whether a word is **present**.

Why does the sparse row give the same answers? The dot product of a row of 0s and 1s with a vector adds up the vector's elements wherever the row has a 1, and ignores the rest (they are multiplied by 0). The sparse row lists exactly those positions. Run the cell to see that the two give the same result.

In [ ]:
tiny_weights = [0.5, -2.0, 0.25, 1.0]

total = 0
for j in sparse_row:
    total += tiny_weights[j]
print(dot(full_row, tiny_weights), total)       # both 0.75

### Activity 6 — One-hot encoding the reviews

We need to build two matrices in the same way, so we write a **function** for it. `one_hot(token_lists)` takes a list of tokenised reviews and returns their sparse matrix: a list with one row per review, where each row is the sorted list of the columns that hold a 1.

A word can occur several times in a review, but its column should only be stored once, so each row is first collected in a **set**. For each word in the review, fill in:

1. the condition: only words that are in our vocabulary have a column (`word_to_index` contains exactly those words)
2. the column for that word

Encoding takes a few seconds.

In [ ]:
def one_hot(token_lists):
    M = []
    for tokens in token_lists:
        row = set()
        for word in tokens:
            if ???:
                row.add(???)
        M.append(sorted(row))       # the columns that hold a 1, in order
    return M

# the tokenised reviews for the training items and for the test items
train_tokens = [tokenized_sents[i] for i in train_ints]
test_tokens = [tokenized_sents[i] for i in test_ints]

M_train = one_hot(train_tokens)
M_test = one_hot(test_tokens)
print(len(M_train), len(M_test))      # the number of rows (reviews) in each matrix: 8000 2000
print(M_train[0][:10])                # the first 10 columns that hold a 1 in the first training review
print([type_list[j] for j in M_train[0][:10]])     # ... and their words

In [ ]:
# ✅ Check
check("8000 training rows and 2000 test rows", lambda: len(M_train) == 8000 and len(M_test) == 2000)
check("the first training review",
      lambda: M_train[0] == sorted(set(word_to_index[w] for w in train_tokens[0] if w in word_to_index)))
check("each column stored once, in order", lambda: all(row == sorted(set(row)) for row in M_train[:100]))
check("every column is between 0 and 4999", lambda: all(0 <= j < 5000 for row in M_test for j in row))

### Activity 7 — How sparse?

Calculate `average_ones`: the average number of 1s in a row of `M_train` (that is, the average number of different vocabulary words in a training review). What proportion of the 5000 columns is that?

In [ ]:
average_ones = ???
print(round(average_ones, 1))                   # you should get 78.5
print(round(average_ones / 5000, 3))            # the proportion of columns that hold a 1

In [ ]:
# ✅ Check
check("average_ones", lambda: abs(average_ones - 78.5) < 0.01)

---
# Part 5 · Matrix-vector multiplication with sparse matrices

At the end of Week 7 we saw that training needs two matrix-vector multiplications:

* **matrix × weights** → a score for every review
* **transpose × errors** → a gradient for every weight

We now write both for our sparse matrices.

### Activity 8 — Scores: sparse matrix × vector

`sparse_mat_vec(M, vector)` does the same job as `mat_vec` in Week 7, but for a sparse matrix. For each row it adds up `vector`'s elements at the columns that hold a 1, just like the tiny example in Part 4. Complete the line inside the inner loop.

In [ ]:
def sparse_mat_vec(M, vector):
    result = []
    for row in M:
        total = 0
        for j in row:
            total += ???
        result.append(total)
    return result

print(sparse_mat_vec([sparse_row], tiny_weights))                    # you should get [0.75]
print(sparse_mat_vec([[0, 2], [1, 2], [3]], tiny_weights))           # you should get [0.75, -1.75, 1.0]

In [ ]:
# ✅ Check
check("one row", lambda: sparse_mat_vec([sparse_row], tiny_weights) == [0.75])
check("three rows", lambda: sparse_mat_vec([[0, 2], [1, 2], [3]], tiny_weights) == [0.75, -1.75, 1.0])
check("same as mat_vec with full rows",
      lambda: sparse_mat_vec([[0, 2], [1, 2]], tiny_weights) == mat_vec([[1, 0, 1, 0], [0, 1, 1, 0]], tiny_weights))

### Activity 9 — Gradients: transpose × vector

For the gradients, Week 5 (and Week 7) used `dot(x[0], errors)`: for each **feature**, multiply its value in each text by that text's error, and add up. With one-hot features the value is 1 or 0, so the gradient for word `j` is simply **the sum of the errors of the reviews that contain word `j`**.

`sparse_transpose_mat_vec(M, vector, num_columns)` calculates this for every column at once. It starts with a 0 for every column, then goes through the rows: for each column `j` that holds a 1 in row `i`, it adds `vector[i]` (row `i`'s error) to column `j`'s total. Complete the line inside the inner loop.

*Example:* with the two reviews `[[0, 2], [1, 2]]` and errors `[10, 1]`, column 0 gets 10 (only review 0 has it), column 1 gets 1, column 2 gets 10 + 1 = 11 (both reviews have it), and column 3 gets 0.

In [ ]:
def sparse_transpose_mat_vec(M, vector, num_columns):
    result = [0] * num_columns
    for i in range(len(M)):
        for j in M[i]:
            result[j] += ???
    return result

print(sparse_transpose_mat_vec([[0, 2], [1, 2]], [10, 1], 4))     # you should get [10, 1, 11, 0]

In [ ]:
# ✅ Check
check("the example", lambda: sparse_transpose_mat_vec([[0, 2], [1, 2]], [10, 1], 4) == [10, 1, 11, 0])
check("same as mat_vec with the transposed full rows",
      lambda: sparse_transpose_mat_vec([[0, 2], [1, 2]], [10, 1], 4) == mat_vec([[1, 0], [0, 1], [1, 1], [0, 0]], [10, 1]))

---
## ✏️ Summary

In this sheet you have turned 10,000 reviews into features a classifier can use:

1. **tokenised** each review into a list of words
2. built a **vocabulary** of the 5000 most frequent words, with a column for each
3. split the reviews into a **training set** and a **test set**
4. **one-hot encoded** each review, storing only the columns that hold a 1 (a **sparse** matrix)
5. written the two matrix-vector multiplications that training needs, for sparse matrices